## 1. Preprocessing Pipeline.
Define a unified preprocessing pipeline for all experiments at 256x256 RGB, compatible with ImageNet-pretrained models.
#### Rationale:
- Standarise inputs across CheXpert (grayscale, variable size) and TBX11K (RGB, 512x512).
- Enable fair comparison among custom CNN and transfer-learning baselines.
- Use ImageNet normalisation to align with pretrained weights.
- Allow an easy switch to 320x320 for the pre-registered sensitivity analysis.

In [12]:
# Define a reproducible preprocessing pipeline for training and validation/testing.
# Baseline: 256x256 RGB for all datasets; optional 320x320 sensitivity via IMG_SIZE variable.

from pathlib import Path
from torchvision import transforms as T

# Project root inferred from this notebook living in notebooks/
PROJECT_ROOT = Path("..").resolve()

# Toggle here for running the sensitivity analysis.
IMG_SIZE = 256 # Set to 320 for the sensitivity analysis.

# ImageNet normalisation (required for the MobileNetV2, EfficientNet-B0, ResNet-50 pretrained weights).
# Rescale pixel values between CheXpert and TBX11K datasets, so they are centred and comparable.
# During that training, inputs were normalised with specific mean and standard deviation values.
# Apply ImageNet normalisation to meet what pretrained filters expect.
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # MobileNetV2, EfficientNet-B0, and ResNet-50 were pretrained on
IMAGENET_STD  = [0.229, 0.224, 0.225] # ImageNet with these exact statistics.

# Training-time augmentations (mild, medically, sensible: avoid aggressive geometric distortions).
train_transforms_chexpert = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),     # Standarise spatial size (IMG_SIZE X IMG_SIZE).
    T.Grayscale(num_output_channels=3), # CheXpert is L (1ch): replicate to RGB (3ch).
    T.RandomHorizontalFlip(p=0.5),      # Lungs are roughly symmetric; safe augmentation.
    T.RandomRotation(degrees=5),        # Small rotations to improve rebustness.
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_transforms_tbx11k = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),   # TBX11K is 512x512 RGB.
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=5),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Validation/test transforms (no stochastic augmentation). Only standarise size, convert to grey scale, to tenson, and normalise to match 
# ImageNet stats.
eval_transforms_chexpert = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transforms_tbx11k = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(f'Preprocessing ready at {IMG_SIZE}x{IMG_SIZE} RGB.')
print(f'- CheXpert: grayscale -> RGB + normalisation.')
print(f'- TBX11K  : RGB + normalisation.')

# Clarify the modelling classes and their data sources (for logging/record-keeping).
print("\nClasses used for training: \n- 'Pneumonia' (CheXpert frontal-only)\n- 'Normal' (TBX11K) \n- 'TB' (TBX11K).")

Preprocessing ready at 256x256 RGB.
- CheXpert: grayscale -> RGB + normalisation.
- TBX11K  : RGB + normalisation.

Classes used for training: 
- 'Pneumonia' (CheXpert frontal-only)
- 'Normal' (TBX11K) 
- 'TB' (TBX11K).
